# STR Tract Analysis

Analysis of STR (Short Tandem Repeat) tracts detected by RPTRF tool and comparison with de novo assembly statistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from Bio import SeqIO
from collections import defaultdict

In [ ]:
# Define paths
REPEAT_REGIONS_DIR = Path("repeatregions")
FASTA_FILE = Path("/home/peterkad/pkadmaster/data/ph/ph_diploid.fa")

In [ ]:
def parse_str_file(file_path, include_homopolymer_info=False):
    """
    Parse a single STR tract file and return list of tract information.
    
    Args:
        file_path: Path to the STR tract file
        include_homopolymer_info: If True, also identify homopolymers
        
    Returns:
        If include_homopolymer_info=False: List of tuples (start, end, length)
        If include_homopolymer_info=True: List of tuples (start, end, length, is_homopolymer)
    """
    tracts = []
    with open(file_path, "r") as file:
        for line in file:
            # Skip header lines
            if (
                line.startswith("*")
                or line.strip() == ""
                or line.startswith("Start")
            ):
                continue
            
            parts = line.split()
            if len(parts) >= 3:
                try:
                    start = int(parts[0])
                    end = int(parts[1])
                    length = int(parts[2])
                    
                    if include_homopolymer_info and len(parts) >= 4:
                        # Extract motif sequence from column 3 (format: "Size(Sequence)")
                        motif_info = parts[3]
                        is_homopolymer = False
                        
                        # Check if motif info contains parentheses
                        if "(" in motif_info and ")" in motif_info:
                            try:
                                # Extract sequence from parentheses
                                _, motif_seq = motif_info.split("(", 1)
                                motif_seq = motif_seq.rstrip(")")
                                
                                # Check if it's a homopolymer (all characters are the same)
                                if len(motif_seq) > 0 and len(set(motif_seq)) == 1:
                                    is_homopolymer = True
                            except:
                                pass
                        
                        tracts.append((start, end, length, is_homopolymer))
                    else:
                        tracts.append((start, end, length))
                except ValueError:
                    # Skip lines that can't be parsed
                    continue
    
    return tracts

In [ ]:
# Find all result-*.txt files in repeatregions folder
str_files = sorted(REPEAT_REGIONS_DIR.glob("result-*.txt"))
print(f"Found {len(str_files)} STR tract files")

In [ ]:
# Parse all STR tract files (with homopolymer info)
all_tracts = []
all_tracts_with_hp = []
tracts_per_file = {}

for str_file in str_files:
    tracts = parse_str_file(str_file, include_homopolymer_info=False)
    tracts_with_hp = parse_str_file(str_file, include_homopolymer_info=True)
    all_tracts.extend(tracts)
    all_tracts_with_hp.extend(tracts_with_hp)
    tracts_per_file[str_file.name] = len(tracts)

print(f"Total STR tracts found: {len(all_tracts)}")

In [ ]:
# Calculate total bases in STR tracts
tract_lengths = [tract[2] for tract in all_tracts]  # Extract length (3rd element)
total_str_bases = sum(tract_lengths)

print(f"Total bases in STR tracts: {total_str_bases:,}")
print(f"Average tract length: {np.mean(tract_lengths):.2f} bases")
print(f"Median tract length: {np.median(tract_lengths):.2f} bases")

In [ ]:
def count_assembly_bases(fasta_path):
    """
    Read FASTA file and return total base count and per-contig sizes.
    
    Args:
        fasta_path: Path to FASTA file
        
    Returns:
        Tuple of (total_bases, contig_sizes_dict)
    """
    total_bases = 0
    contig_sizes = {}
    
    for record in SeqIO.parse(fasta_path, "fasta"):
        seq_length = len(record.seq)
        total_bases += seq_length
        contig_sizes[record.id] = seq_length
    
    return total_bases, contig_sizes

In [ ]:
# Read FASTA file and calculate total assembly size
total_assembly_bases, contig_sizes = count_assembly_bases(FASTA_FILE)

print(f"Total assembly size: {total_assembly_bases:,} bases")
print(f"Number of contigs: {len(contig_sizes)}")
print(f"Average contig size: {np.mean(list(contig_sizes.values())):,.2f} bases")
print(f"Median contig size: {np.median(list(contig_sizes.values())):,.2f} bases")

In [ ]:
# Calculate descriptive statistics
str_coverage_percent = (total_str_bases / total_assembly_bases) * 100

# Calculate bp per 1 repeat region: average number of reference basepairs per repeat region
# This tells us the spacing/density of repeat regions in the genome
bp_per_repeat_region = total_assembly_bases / len(all_tracts)

stats = {
    "Total STR tracts": len(all_tracts),
    "Total STR bases": f"{total_str_bases:,}",
    "Total assembly bases": f"{total_assembly_bases:,}",
    "STR coverage (%)": f"{str_coverage_percent:.4f}",
    "bp per 1 repeat region": f"{bp_per_repeat_region:.2f}",
    "Mean tract length": f"{np.mean(tract_lengths):.2f}",
    "Median tract length": f"{np.median(tract_lengths):.2f}",
    "Min tract length": f"{np.min(tract_lengths)}",
    "Max tract length": f"{np.max(tract_lengths)}",
    "Std dev tract length": f"{np.std(tract_lengths):.2f}",
    "Number of files processed": len(str_files)
}

# Create summary DataFrame
summary_df = pd.DataFrame([stats]).T
summary_df.columns = ["Value"]
print("\n=== Summary Statistics ===")
print(summary_df.to_string())

In [ ]:
# Create histogram of tract lengths
plt.figure(figsize=(12, 6))
plt.hist(tract_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Tract Length (bases)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of STR Tract Lengths', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print quartiles
q25, q50, q75 = np.percentile(tract_lengths, [25, 50, 75])
print(f"\nTract Length Quartiles:")
print(f"  25th percentile: {q25:.2f} bases")
print(f"  50th percentile (median): {q50:.2f} bases")
print(f"  75th percentile: {q75:.2f} bases")
print(f"  IQR: {q75 - q25:.2f} bases")

In [ ]:
# Create a log-scale histogram for better visualization of the distribution
plt.figure(figsize=(12, 6))
plt.hist(tract_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Tract Length (bases)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of STR Tract Lengths (Log Scale)', fontsize=14, fontweight='bold')
plt.yscale('log')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Create DataFrame for tract data
tract_df = pd.DataFrame(all_tracts, columns=['Start', 'End', 'Length'])

# Display summary statistics table
print("=== Detailed Tract Statistics ===")
print(tract_df['Length'].describe())

In [ ]:
# Visualize tracts per file distribution
tracts_per_file_df = pd.DataFrame(list(tracts_per_file.items()), columns=['File', 'Tract_Count'])
tracts_per_file_df = tracts_per_file_df.sort_values('Tract_Count', ascending=False)

plt.figure(figsize=(14, 6))
plt.bar(range(len(tracts_per_file_df)), tracts_per_file_df['Tract_Count'], alpha=0.7)
plt.xlabel('File Index (sorted by tract count)', fontsize=12)
plt.ylabel('Number of Tracts', fontsize=12)
plt.title('Number of STR Tracts per File', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFiles with most tracts:")
print(tracts_per_file_df.head(10).to_string(index=False))
print(f"\nFiles with fewest tracts:")
print(tracts_per_file_df.tail(10).to_string(index=False))